# Train SDXL LoRA for Bonnie (West Highland White Terrier)

In [1]:
import os
import shutil
import subprocess
import glob
import zipfile

# 1. Install dependencies and download the script
subprocess.run(["pip", "install", "-q", "torchao>=0.16.0", "bitsandbytes"])
subprocess.run(["wget", "-q", "-O", "train_dreambooth_lora_sdxl.py", "https://raw.githubusercontent.com/huggingface/diffusers/v0.37.1/examples/dreambooth/train_dreambooth_lora_sdxl.py"])

# 2. Reset the working directories
shutil.rmtree("/kaggle/working/dataset", ignore_errors=True)
os.makedirs("/kaggle/working/dataset/processed", exist_ok=True)
os.makedirs("/kaggle/working/dataset/class_images", exist_ok=True)

# 3. Foolproof extraction and copying
input_dir = "/kaggle/input/bonnie-dataset"
zip_files = glob.glob(f"{input_dir}/*.zip")

if zip_files:
    print(f"Found archive. Extracting {zip_files[0]} directly...")
    with zipfile.ZipFile(zip_files[0], 'r') as zip_ref:
        zip_ref.extractall("/kaggle/working/")
else:
    print("Archive already extracted. Searching for images...")
    for root, dirs, files in os.walk(input_dir):
        for f in files:
            if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                src = os.path.join(root, f)
                # Route files to the correct folder based on their original path
                if "class" in root.lower() or "class" in f.lower():
                    shutil.copy(src, "/kaggle/working/dataset/class_images")
                else:
                    shutil.copy(src, "/kaggle/working/dataset/processed")

print(f"Ready. Instance images: {len(os.listdir('/kaggle/working/dataset/processed'))}")
print(f"Ready. Class images: {len(os.listdir('/kaggle/working/dataset/class_images'))}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 47.4 MB/s eta 0:00:00
Archive already extracted. Searching for images...
Ready. Instance images: 0
Ready. Class images: 0


In [2]:
import accelerate
import os

config_path = os.path.expanduser('~/.cache/huggingface/accelerate/default_config.yaml')
os.makedirs(os.path.dirname(config_path), exist_ok=True)
config_content = '''compute_environment: LOCAL_MACHINE
debug: false
distributed_type: 'NO'
downcast_bf16: 'no'
gpu_ids: all
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 1
rdzv_backend: static
same_network: true
use_cpu: false
'''
with open(config_path, 'w') as f:
    f.write(config_content)

In [3]:
!accelerate launch train_dreambooth_lora_sdxl.py \
  --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" \
  --instance_data_dir="dataset/processed" \
  --instance_prompt="bonnie_dog" \
  --with_prior_preservation \
  --prior_loss_weight=1.0 \
  --class_data_dir="dataset/class_images" \
  --class_prompt="a photo of a west highland white terrier dog" \
  --dataloader_num_workers=0 \
  --resolution=1024 \
  --train_batch_size=1 \
  --sample_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --gradient_checkpointing \
  --learning_rate=1e-4 \
  --lr_scheduler="cosine_with_restarts" \
  --lr_warmup_steps=50 \
  --max_train_steps=800 \
  --checkpointing_steps=400 \
  --seed=42 \
  --output_dir="bonnie-lora-sdxl" \
  --mixed_precision="fp16" \
  --prior_generation_precision="fp16" \
  --use_8bit_adam \
  --rank=32 \
  --snr_gamma=5.0 \
  --report_to="tensorboard"

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
09/21/2026 05:11:05 - INFO - __main__ - [RANK 0] Distributed environment: DistributedType.NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: fp16

09/21/2026 05:11:05 - INFO - httpx - HTTP Request: GET https://huggingface.co/api/models/stabilityai/stable-diffusion-xl-base-1.0 "HTTP/1.1 200 OK"
09/21/2026 05:11:05 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main